# Module 6 Lab: Fairness Evaluation on the Titanic Model

**Practical Machine Learning Foundations**

**Purpose:** audit the Titanic classifier for bias by disaggregating every metric by sex and class, measuring where its errors fall, and explaining its decisions with SHAP, LIME, permutation importance, and partial dependence plots.

**Date:** 2026-08-24 | **Author:** Nick Garner

We built this model, evaluated it, and attacked it. One question remains, and it is the one that decides whether a model should ship: **is it fair, and can you explain what it does?**

Those two questions are inseparable. If you cannot explain why a model made a prediction, you cannot audit it for fairness, debug it when it fails, or defend it when someone challenges it.

### A note on this dataset

Titanic is a teaching dataset about a real disaster in which 1,500 people died, and the survival patterns in it are not statistical curiosities. They reflect the evacuation norms of 1912: women and children were loaded into lifeboats first, and first class passengers were berthed nearer the boat deck. When our model learns that gender predicts survival, it has learned a historical social policy.

That makes it an unusually honest fairness case study, because the bias in the data is undeniable and well documented rather than hidden. The question the whole module turns on is this: **when a model faithfully reproduces a pattern from the past, is it accurate or is it biased?** Often it is both.

### What you will do

| Section | Focus |
|---|---|
| 1. Where bias enters | Mapping the pipeline stages onto this dataset |
| 2. Grouped accuracy | Disaggregating by sex and class |
| 3. False negative disparities | Where the errors actually fall |
| 4. Formal fairness metrics | And why you cannot satisfy them all |
| 5. Proxy detection | Why deleting a column does not work |
| 6. SHAP | Local and global explanations |
| 7. LIME | And its stability problem |
| 8. Permutation importance | Which features the model relies on |
| 9. PDP and ICE | How those features shape predictions |
| 10. Mitigation | And documenting what you chose |
| 11. Where you go next | Career tracks, portfolio, certifications |


## Section 0: Setup and rebuilding the model

Same data and the same preprocessing pipeline as the previous modules. We train two models: a logistic regression for interpretability and a gradient boosting model because tree-based SHAP is fast and it gives us a second opinion.


In [ ]:
# LIME not preinstalled
! pip install lime

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)
print("Setup complete.")


In [ ]:
URL = ("https://raw.githubusercontent.com/datasciencedojo/"
       "datasets/master/titanic.csv")
try:
    raw = pd.read_csv(URL)
    print("Loaded the real Titanic dataset.")
except Exception as e:
    print("Network unavailable (" + type(e).__name__ + "), building a stand-in.")
    n = 891
    pclass = rng.choice([1, 2, 3], n, p=[.24, .21, .55])
    sex = rng.choice(["male", "female"], n, p=[.65, .35])
    age = np.clip(np.where(pclass == 1, rng.normal(38, 14, n),
                           rng.normal(27, 13, n)), 0.5, 80).round(1)
    age[rng.choice(n, 177, replace=False)] = np.nan
    base = np.where(sex == "female", .74, .19) * np.where(pclass == 1, 1.25, .9)
    raw = pd.DataFrame({
        "PassengerId": np.arange(1, n + 1), "Pclass": pclass, "Sex": sex,
        "Survived": rng.binomial(1, np.clip(base, 0, 1)),
        "Name": ["Doe, Mr. P" + str(i) for i in range(n)], "Age": age,
        "SibSp": rng.choice([0, 1, 2, 3], n, p=[.68, .23, .06, .03]),
        "Parch": rng.choice([0, 1, 2], n, p=[.76, .13, .11]),
        "Ticket": ["T" + str(x) for x in rng.integers(1000, 9999, n)],
        "Fare": np.where(pclass == 1, rng.lognormal(4.2, .7, n),
                         rng.lognormal(2.5, .6, n)).round(4),
        "Cabin": np.where(rng.random(n) < .23, "C85", None),
        "Embarked": rng.choice(["S", "C", "Q"], n, p=[.72, .19, .09])})

df = raw.dropna(subset=["Embarked"]).reset_index(drop=True)
df["Deck"] = df["Cabin"].str[0].fillna("Unknown")
df["Had_cabin_record"] = df["Cabin"].notna().astype(int)
df["Age_was_missing"] = df["Age"].isnull().astype(int)
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1

NUMERIC = ["Age", "Fare", "SibSp", "Parch", "FamilySize"]
CATEGORICAL = ["Sex", "Embarked", "Deck"]
PASSTHROUGH = ["Pclass", "Age_was_missing", "Had_cabin_record"]
FEATURES = NUMERIC + CATEGORICAL + PASSTHROUGH

X = df[FEATURES]
y = df["Survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=RANDOM_STATE)

print("Train:", X_train.shape, "| Test:", X_test.shape)
print("Overall survival rate:", round(y.mean(), 3))


In [ ]:
def make_preprocessor():
    return ColumnTransformer([
        ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                          ("scale", StandardScaler())]), NUMERIC),
        ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                          ("encode", OneHotEncoder(handle_unknown="ignore",
                                                   sparse_output=False))]),
         CATEGORICAL),
        ("pass", "passthrough", PASSTHROUGH)])


logreg = Pipeline([("preprocess", make_preprocessor()),
                   ("model", LogisticRegression(max_iter=1000,
                                                random_state=RANDOM_STATE))])
gbm = Pipeline([("preprocess", make_preprocessor()),
                ("model", GradientBoostingClassifier(
                    random_state=RANDOM_STATE))])

logreg.fit(X_train, y_train)
gbm.fit(X_train, y_train)

print("Logistic regression test accuracy:", round(logreg.score(X_test, y_test), 4))
print("Gradient boosting test accuracy:  ", round(gbm.score(X_test, y_test), 4))
print()
print("Both look fine. That single number is exactly the problem.")


---
# Section 1: Where bias enters

Bias is not a bug in one stage of the pipeline. It can enter at every stage, which is why there is no single place to go and fix it. Three kinds are worth separating:

- **Statistical bias**: systematic error, where a model consistently over- or under-predicts for a group
- **Societal bias**: historical inequities encoded in the data, so a model trained on decades of discriminatory hiring learns those patterns
- **Algorithmic bias**: the model amplifies existing disparities, because the optimization objective can worsen group-level outcomes

Almost none of it is intentional. Nobody sets out to build a discriminatory model, and **"we just trained on the data" is not a defense** when the data encodes historical discrimination.

### The six stages, mapped onto this dataset

| Stage | The general risk | On Titanic specifically |
|---|---|---|
| Problem definition | Whose values define "success"? | We chose to predict survival, which encodes a 1912 evacuation policy as the target |
| Data collection | Selection bias (who is in the data), measurement bias (how features and labels were captured), <br>representation bias (does it reflect reality), historical bias, temporal bias | Records are incomplete for third class, and `Cabin` is 77% missing precisely because clerks recorded less about poorer passengers |
| Feature engineering | Proxy variables encode protected attributes | `Fare` and `Deck` are wealth proxies; `Name` titles encode both sex and marital status |
| Model training | Optimization bias, where aggregate objectives mask subgroup disparity, <br>plus class imbalance leaving small groups treated as noise | The optimizer minimizes total error, and third class is the majority so its errors dominate |
| Evaluation | Standard splits do not stratify by protected group | Our test set has never once been examined by sex or class |
| Deployment | Feedback loops turn biased outputs into biased future data | Predictive policing is the canonical case: more officers produce more arrests, which the data records as more crime |

Two other sources:
**Labeling bias**: human annotators bring their own assumptions, ambiguous cases get resolved by cultural frame, and majority-vote label aggregation silences minority perspectives, so what we call "ground truth" is often a subjective judgment rather than an objective fact. And **aggregation bias**: fitting one model to diverse populations when different subgroups genuinely need different parameters.


In [ ]:
base_rates = pd.crosstab(df["Sex"], df["Pclass"], values=df["Survived"],
                         aggfunc="mean").round(3)
base_rates.columns = ["1st class", "2nd class", "3rd class"]
print("Actual survival rate by sex and class (the historical record):")
print(base_rates)
print()
print("Survival rate by sex overall:")
print(df.groupby("Sex")["Survived"].mean().round(3))
print()
print("A first class woman was several times more likely to survive than a")
print("third class man. The model is about to learn that, faithfully.")


**Base rates differ enormously between groups.** In Section 4 it turns out to be the mathematical reason fairness is impossible to fully achieve.

---
# Section 2: Grouped accuracy by sex and class

Overall accuracy is a weighted average. A model can post 95% overall while performing at 70% for a minority group, and no standard metric will flag it. **Disaggregation**: compute every metric separately for every group you care about.


In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, roc_auc_score)

audit = X_test.copy()
audit["actual"] = y_test.values
audit["predicted"] = logreg.predict(X_test)
audit["probability"] = logreg.predict_proba(X_test)[:, 1]

def group_metrics(frame, group_col):
    rows = []
    for name, g in frame.groupby(group_col):
        cm = confusion_matrix(g["actual"], g["predicted"], labels=[0, 1])
        tn, fp, fn, tp = cm.ravel()
        rows.append({
            group_col: name,
            "n": len(g),
            "base rate": round(g["actual"].mean(), 3),
            "accuracy": round(accuracy_score(g["actual"], g["predicted"]), 3),
            "precision": round(precision_score(g["actual"], g["predicted"],
                                               zero_division=0), 3),
            "recall (TPR)": round(recall_score(g["actual"], g["predicted"],
                                               zero_division=0), 3),
            "FNR": round(fn / (fn + tp), 3) if (fn + tp) else np.nan,
            "FPR": round(fp / (fp + tn), 3) if (fp + tn) else np.nan,
        })
    return pd.DataFrame(rows).set_index(group_col)


print("OVERALL accuracy:", round(accuracy_score(audit["actual"],
                                                audit["predicted"]), 3))
print()
print("Disaggregated by sex:")
print(group_metrics(audit, "Sex"))


In [ ]:
print("Disaggregated by passenger class:")
print(group_metrics(audit, "Pclass"))
print()
audit["subgroup"] = audit["Sex"] + ", class " + audit["Pclass"].astype(str)
print("Disaggregated by both (intersectional view):")
print(group_metrics(audit, "subgroup").sort_values("accuracy"))


Looking at sex alone or class alone can each look tolerable while a specific **combination** of the two is served far worse than average. This is exactly the finding of the Gender Shades study, which reported error rates as high as 34.7% for darker-skinned women against 0.8% for lighter-skinned men in commercial facial recognition, a disparity invisible in either single-axis breakdown.

Note also that accuracy is not the whole story here: the groups have very different **base rates**, so an accuracy figure for a group where 90% of people share one outcome is easy to achieve and means little.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

by_sub = group_metrics(audit, "subgroup").sort_values("accuracy")
by_sub[["accuracy", "recall (TPR)"]].plot(kind="barh", ax=axes[0],
                                          color=["steelblue", "darkorange"])
axes[0].axvline(accuracy_score(audit["actual"], audit["predicted"]),
                color="firebrick", linestyle="--", label="overall accuracy")
axes[0].set_title("Performance by subgroup")
axes[0].set_xlabel("Score"); axes[0].legend(fontsize=8)

by_sub[["FNR", "FPR"]].plot(kind="barh", ax=axes[1],
                            color=["firebrick", "steelblue"])
axes[1].set_title("Error rates by subgroup")
axes[1].set_xlabel("Rate")
plt.tight_layout(); plt.show()


---
# Section 3: Disparities in false negatives

Accuracy treats all errors alike. Fairness analysis almost never can, because the two error types land on different people and carry different consequences.

For this model, a **false negative** means predicting someone died when they actually survived. Reframe it as the deployment decision it stands in for, such as a triage or resource allocation model, and a false negative is a person the system failed to flag for help. That is the error that harms the individual.


In [ ]:
print("FALSE NEGATIVE ANALYSIS")
print("=" * 66)
print("A false negative here = the model said 'did not survive' about")
print("someone who actually did. In a deployed triage system, that is a")
print("person the model overlooked.\n")

for col in ["Sex", "Pclass"]:
    m = group_metrics(audit, col)
    print("By " + col + ":")
    for name, row in m.iterrows():
        n_fn = int(round(row["FNR"] * row["base rate"] * row["n"]))
        print("  " + str(name).ljust(10),
              "FNR", str(row["FNR"]).ljust(7),
              "| missed", str(n_fn).rjust(2), "of",
              int(round(row["base rate"] * row["n"])), "actual survivors")
    print()

fnr_by_sex = group_metrics(audit, "Sex")["FNR"]
print("FNR disparity ratio (male / female):",
      round(fnr_by_sex.max() / max(fnr_by_sex.min(), 1e-9), 2))


That ratio is the headline number of a fairness audit. A model whose false negative rate is several times higher for one group is systematically failing that group, and the overall accuracy figure conceals it.

This is the COMPAS finding. ProPublica's 2016 investigation of the recidivism scoring tool used in bail, sentencing, and parole decisions found that Black defendants were falsely flagged as high risk at 45% against 23% for white defendants.

### Per-group confusion matrices

The disaggregated confusion matrix shows exactly where each group's errors sit.


In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

groups = sorted(audit["Sex"].unique())
fig, axes = plt.subplots(1, len(groups) + 1, figsize=(5 * (len(groups) + 1), 4))

ConfusionMatrixDisplay.from_predictions(
    audit["actual"], audit["predicted"], normalize="true",
    display_labels=["died", "survived"], cmap="Blues", ax=axes[0],
    colorbar=False, values_format=".2f")
axes[0].set_title("Everyone")

for ax, g in zip(axes[1:], groups):
    sub = audit[audit["Sex"] == g]
    ConfusionMatrixDisplay.from_predictions(
        sub["actual"], sub["predicted"], normalize="true",
        display_labels=["died", "survived"], cmap="Blues", ax=ax,
        colorbar=False, values_format=".2f")
    ax.set_title(g + " (n=" + str(len(sub)) + ")")
plt.suptitle("Row-normalized: the bottom-left cell of each is that group's FNR",
             fontsize=11)
plt.tight_layout(); plt.show()


---
# Section 4: Formal fairness metrics, and why you cannot have them all

The fairness literature contains at least 21 formal definitions. Five come up constantly:

- **Demographic parity**: each group receives positive predictions at the same rate. If 30% of men are approved, 30% of women should be.
- **Equalized odds**: true positive rate and false positive rate are equal across groups, so the model is equally accurate for each regardless of base rates.
- **Predictive parity**: precision is equal across groups. When the model says positive, it is right equally often for everyone.
- **Individual fairness**: similar individuals receive similar outcomes. Defining "similar" is itself a value judgment.
- **Counterfactual fairness**: would this person get the same decision if only their group membership changed?

Every one of these is reasonable and defensible. Let's measure all of them at once.


In [ ]:
def fairness_report(frame, group_col, threshold=0.5):
    rows = []
    for name, g in frame.groupby(group_col):
        pred = (g["probability"] >= threshold).astype(int)
        cm = confusion_matrix(g["actual"], pred, labels=[0, 1])
        tn, fp, fn, tp = cm.ravel()
        rows.append({
            group_col: name,
            "selection rate": round(pred.mean(), 3),          # demographic parity
            "TPR": round(tp / (tp + fn), 3) if (tp + fn) else np.nan,
            "FPR": round(fp / (fp + tn), 3) if (fp + tn) else np.nan,
            "precision (PPV)": round(tp / (tp + fp), 3) if (tp + fp) else np.nan,
            "base rate": round(g["actual"].mean(), 3),
        })
    return pd.DataFrame(rows).set_index(group_col)


fr = fairness_report(audit, "Sex")
print(fr)
print()
print("DEFINITION-BY-DEFINITION VERDICT")
print("=" * 60)
print("Demographic parity  gap:", round(fr['selection rate'].max() -
                                        fr['selection rate'].min(), 3),
      "  (0.0 would be perfect parity)")
print("Equalized odds      TPR gap:", round(fr['TPR'].max() - fr['TPR'].min(), 3),
      "| FPR gap:", round(fr['FPR'].max() - fr['FPR'].min(), 3))
print("Predictive parity   PPV gap:", round(fr['precision (PPV)'].max() -
                                            fr['precision (PPV)'].min(), 3))


### The impossibility theorem, demonstrated

Chouldechova (2017) proved that when base rates differ between groups, you **cannot** simultaneously achieve equal false positive rates, equal false negative rates, and equal positive predictive values. Kleinberg and colleagues (2016) showed that calibration and equal error rates are mathematically incompatible unless base rates are identical or prediction is perfect.

Our base rates differ enormously, so we are squarely inside the theorem's conditions. Watch what happens when we try to fix one definition by adjusting the decision threshold for each group.


In [ ]:
def metrics_at(frame, group, threshold):
    g = frame[frame["Sex"] == group]
    pred = (g["probability"] >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(g["actual"], pred, labels=[0, 1]).ravel()
    return {"selection": pred.mean(),
            "TPR": tp / (tp + fn) if (tp + fn) else np.nan,
            "FPR": fp / (fp + tn) if (fp + tn) else np.nan,
            "PPV": tp / (tp + fp) if (tp + fp) else np.nan}


grid = np.linspace(0.02, 0.98, 97)
female_t, male_t = 0.5, 0.5

# Strategy A: tune male threshold to equalize SELECTION RATE (demographic parity)
target = metrics_at(audit, "female", female_t)["selection"]
male_t_dp = min(grid, key=lambda t: abs(metrics_at(audit, "male", t)["selection"]
                                        - target))
# Strategy B: tune male threshold to equalize TPR (equalized odds)
target_tpr = metrics_at(audit, "female", female_t)["TPR"]
male_t_eo = min(grid, key=lambda t: abs(metrics_at(audit, "male", t)["TPR"]
                                        - target_tpr))

rows = []
for label, mt in [("Equal thresholds (0.5)", 0.5),
                  ("Tuned for demographic parity", male_t_dp),
                  ("Tuned for equalized odds (TPR)", male_t_eo)]:
    f = metrics_at(audit, "female", female_t)
    m = metrics_at(audit, "male", mt)
    rows.append({
        "strategy": label,
        "male threshold": round(mt, 2),
        "selection gap": round(abs(f["selection"] - m["selection"]), 3),
        "TPR gap": round(abs(f["TPR"] - m["TPR"]), 3),
        "FPR gap": round(abs(f["FPR"] - m["FPR"]), 3),
        "PPV gap": round(abs(f["PPV"] - m["PPV"]), 3)})
pd.DataFrame(rows).set_index("strategy")


Read across each row. Every strategy drives **one** gap toward zero and leaves the others open, and closing one tends to widen another. That is not a bug in our tuning loop; it is the theorem. When base rates genuinely differ, the definitions are mutually incompatible.

This is why both sides of the COMPAS argument were mathematically correct. ProPublica measured false positive rates by race and found a large disparity. Northpointe measured predictive accuracy by race and found it roughly equal at about 60%. Both were right, because the system satisfied one definition while violating another, and no amount of engineering could have satisfied both.

**The consequence for practitioners: choosing a fairness definition is a values decision, not a technical one.** Different stakeholders have legitimate reasons to prefer different ones. A defendant wants a low false positive rate and not to be wrongly flagged; society may prioritize a low false negative rate and not missing real risk. There is no formula that resolves that, and pretending the choice is technical hides it from the people entitled to weigh in.

### The fairness-accuracy tradeoff

Enforcing a fairness constraint usually costs some accuracy. Let's measure it rather than argue about it.


In [ ]:
pred_fair = np.where(audit["Sex"] == "male",
                     (audit["probability"] >= male_t_eo).astype(int),
                     (audit["probability"] >= female_t).astype(int))

print("Standard model (threshold 0.5 for everyone):")
print("  accuracy", round(accuracy_score(audit["actual"], audit["predicted"]), 4),
      "| TPR gap", round(abs(fr['TPR'].diff().iloc[-1]), 3))
print("Group-adjusted thresholds (equalized odds):")
print("  accuracy", round(accuracy_score(audit["actual"], pred_fair), 4),
      "| TPR gap", round(rows[2]["TPR gap"], 3))
print()
print("Accuracy cost:", round((accuracy_score(audit["actual"],
                                              audit["predicted"]) -
                                accuracy_score(audit["actual"], pred_fair))
                               * 100, 1), "percentage points")


To give male passengers the same true positive rate as female passengers, the threshold for men has to fall so low that the model predicts survival for nearly all of them, which wrecks overall accuracy. The gap closed and the model got much worse.

That is the fairness-accuracy tradeoff, and it is severe here *because* the base rates are so far apart. Do not read this one result as "fairness always costs thirty points"; read it as "the cost scales with how different the groups are, so measure it rather than assuming it."

Before accepting "fairness reduces accuracy" as an objection, notice the counter-argument: **accuracy measured on biased data is not true accuracy.** A model that is 95% accurate at reproducing historical discrimination is accurately biased, not genuinely good. Fairness constraints can also improve robustness by forcing a model to work across subgroups rather than overfitting to the majority, and fair systems build more trust over time. There is a business case, not only an ethical one.

One caution on the technique above: **using different thresholds per group is itself legally fraught** in many jurisdictions, since it means explicitly treating people differently based on a protected attribute. It is a useful diagnostic, and a contested remedy.

---
# Section 5: Proxy detection, or why deleting the column does not work

The intuitive fix is "just remove sex from the model." This is called **fairness through unawareness**, and it fails, because other features carry the same information.


In [ ]:
FEATURES_BLIND = [f for f in FEATURES if f != "Sex"]

def make_blind_preprocessor():
    return ColumnTransformer([
        ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                          ("scale", StandardScaler())]), NUMERIC),
        ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                          ("encode", OneHotEncoder(handle_unknown="ignore",
                                                   sparse_output=False))]),
         ["Embarked", "Deck"]),
        ("pass", "passthrough", PASSTHROUGH)])


blind = Pipeline([("preprocess", make_blind_preprocessor()),
                  ("model", LogisticRegression(max_iter=1000,
                                               random_state=RANDOM_STATE))])
blind.fit(X_train[FEATURES_BLIND], y_train)

blind_audit = audit.copy()
blind_audit["predicted"] = blind.predict(X_test[FEATURES_BLIND])

print("Model WITH sex, FNR by sex:")
print(" ", dict(group_metrics(audit, "Sex")["FNR"]))
print("Model WITHOUT sex, FNR by sex:")
print(" ", dict(group_metrics(blind_audit, "Sex")["FNR"]))
print()
print("Accuracy with sex:   ", round(accuracy_score(y_test, audit["predicted"]), 4))
print("Accuracy without sex:", round(accuracy_score(y_test,
                                                    blind_audit["predicted"]), 4))


In [ ]:
# How much sex information survives in the remaining features?
sex_target = (X_train["Sex"] == "female").astype(int)
proxy_detector = Pipeline([("preprocess", make_blind_preprocessor()),
                           ("model", RandomForestClassifier(
                               n_estimators=300, random_state=RANDOM_STATE))])
proxy_detector.fit(X_train[FEATURES_BLIND], sex_target)

sex_test = (X_test["Sex"] == "female").astype(int)
proxy_auc = roc_auc_score(sex_test,
                          proxy_detector.predict_proba(
                              X_test[FEATURES_BLIND])[:, 1])

print("Predicting SEX from the 'sex-blind' feature set alone:")
print("  accuracy:", round(proxy_detector.score(X_test[FEATURES_BLIND],
                                                sex_test), 4))
print("  AUC:     ", round(proxy_auc, 4), "(0.5 would mean no information)")
print()
print("Well above chance, so sex information does leak through the other")
print("features. But the leak is partial: the proxies on Titanic are weak,")
print("which turns out to matter enormously.")



Removing `Sex` **did** substantially close the false negative gap here, at a large cost in accuracy. That is the opposite of what "fairness through unawareness does not work" usually predicts, and the reason is measurable: the proxy detector recovers sex at only moderate AUC, so on this dataset the protected attribute carried signal that little else duplicated. Delete it and the information largely does go away.

The general lesson: **whether unawareness works depends entirely on how strong your proxies are, and that is an empirical question you must measure rather than assume.** Titanic has weak proxies. Real deployment datasets usually have strong ones, which is why the technique fails in the documented cases below. The transferable habit is the proxy detector you just built: before assuming a column is safe to drop, train a model to predict the protected attribute from everything else and look at the AUC.

Note also what unawareness cost even here: a large drop in accuracy, and it did nothing for the disparity across passenger class, a protected-adjacent attribute we left in the model.

This is the central lesson of the real-world cases:

- The **Optum healthcare algorithm** used healthcare spending as a proxy for health need. Black patients historically had less access and therefore lower spending at the same level of illness, so at equal risk scores they were significantly sicker. It affected roughly 200 million patients a year and cut the number of Black patients flagged for extra care by about half. No protected attribute appeared in the model.
- **Lending models** recreated redlining digitally through zip code, shopping patterns, and social networks. The Apple Card case in 2019 saw women given far lower credit limits than husbands with shared finances.
- **Word embeddings** trained on internet text encode stereotypes directly, with "man is to computer programmer as woman is to homemaker" a documented embedding relationship.
- **Amazon's recruiting tool** was trained on ten years of mostly male resumes and learned to penalize mentions of women's colleges and women's organizations.
- The **Allegheny Family Screening Tool** used government service usage data, effectively penalizing poverty rather than predicting neglect.

Proxies are everywhere: zip code encodes race, names encode gender and ethnicity, spending encodes access, and interaction effects create proxies that no single feature reveals. **Historical data is not objective; it reflects the power structures that produced it.**


---
# Section 6: SHAP

Fairness metrics tell you **that** a model treats groups differently. Explainability tells you **why**. SHAP is the closest thing the field has to a gold standard for that second question.

**SHAP** stands for SHapley Additive exPlanations, and it is built on Shapley values from cooperative game theory, a concept from 1953. The game theory analogy: three people collaborate to earn $100, and the Shapley value divides the payout fairly by measuring each person's average marginal contribution across every possible joining order. Translated to ML, the "project" is a prediction and the "people" are features.

Two properties make it the default choice:

1. **It is mathematically principled.** The Shapley value is the unique solution satisfying four axioms: efficiency, symmetry, dummy, and additivity.
2. **The contributions add up exactly.** base value + sum of SHAP values = the model's output. If the model predicts 0.8 and the average prediction is 0.5, the SHAP values account for exactly where that 0.3 came from.

It works with any model, and there are fast specialized implementations: `TreeExplainer` for tree models, `DeepExplainer` for neural networks, and `KernelExplainer` as the slower universal fallback.


In [ ]:
import shap

# The pipeline's preprocessor turns our DataFrame into a numeric matrix.
# SHAP explains the model that consumes that matrix, so transform first.
pre = gbm.named_steps["preprocess"]
model_only = gbm.named_steps["model"]

X_train_t = pre.transform(X_train)
X_test_t = pre.transform(X_test)
feat_names = [n.split("__")[-1] for n in pre.get_feature_names_out()]

explainer = shap.TreeExplainer(model_only)
shap_values = explainer(X_test_t)
shap_values.feature_names = feat_names

print("SHAP values computed:", shap_values.values.shape,
      "(one value per feature per prediction)")
print()
i = 0
pred_i = gbm.predict_proba(X_test.iloc[[i]])[0, 1]
print("Additivity check on passenger", i)
print("  base value + sum of SHAP values =",
      round(float(shap_values.base_values[i] +
                  shap_values.values[i].sum()), 4))
print("  model's raw output (log-odds)   =",
      round(float(np.log(pred_i / (1 - pred_i))), 4))


### Local explanations: why this prediction?

A **waterfall plot** stacks each feature's contribution from the base value to the final prediction. This is the plot to reach for when someone asks "why did the model decide that about me?"


In [ ]:
passenger = X_test.iloc[i]
print("Passenger:", dict(passenger[["Sex", "Pclass", "Age", "Fare"]]))
print("Actual outcome:", "survived" if y_test.iloc[i] == 1 else "did not survive")
print("Model's predicted survival probability:", round(pred_i, 3))
print()
shap.plots.waterfall(shap_values[i], max_display=10, show=False)
plt.tight_layout(); plt.show()


Red bars push the prediction up, blue bars pull it down, and the numbers on the left are the feature values for this specific passenger. That is a complete, auditable account of one decision.

For regulatory purposes this is not optional. GDPR's right to explanation and the adverse action notice requirements under the Equal Credit Opportunity Act (ECOA) both demand individual-level explanations of automated decisions. A waterfall plot is a defensible answer; "the model said so" is not.

### Global explanations: what drives the model overall?

The **summary plot** puts one dot per sample per feature, positioned by SHAP value and coloured by the feature's value. It shows importance and direction at the same time.


In [ ]:
shap.plots.beeswarm(shap_values, max_display=12, show=False)
plt.title("SHAP summary: which features matter, and in which direction")
plt.tight_layout(); plt.show()


In [ ]:
shap.plots.bar(shap_values, max_display=12, show=False)
plt.title("Mean absolute SHAP value: the quickest global importance ranking")
plt.tight_layout(); plt.show()


### SHAP for fairness auditing

Compare SHAP contributions **across protected groups**: if a feature contributes far more for one group than another, that is a proxy discrimination signal.


In [ ]:
sex_mask = (X_test["Sex"] == "female").values
sv = shap_values.values

comparison = pd.DataFrame({
    "mean |SHAP| female": np.abs(sv[sex_mask]).mean(axis=0),
    "mean |SHAP| male": np.abs(sv[~sex_mask]).mean(axis=0),
}, index=feat_names)
comparison["ratio"] = (comparison["mean |SHAP| female"] /
                       comparison["mean |SHAP| male"].replace(0, np.nan))
comparison = comparison.sort_values("ratio", ascending=False).round(4)

print("Feature contribution by group (ratios far from 1.0 warrant a look):")
print(comparison.head(8))
print()
print("Mean SIGNED SHAP for the sex features, showing direction:")
for f in [n for n in feat_names if n.startswith("Sex")]:
    j = feat_names.index(f)
    print(" ", f.ljust(12), round(float(sv[:, j].mean()), 4))


SHAP interaction values go further still, revealing hidden feature-by-group interactions that a flat importance ranking would never surface. And for documentation, these plots turn a model card from a list of feature names into concrete evidence of what the model actually relies on.

---
# Section 7: LIME

LIME takes a completely different route to the same goal. **Local Interpretable Model-agnostic Explanations**, published by Ribeiro, Singh, and Guestrin in 2016, rests on one insight: even if a model is globally complex, its behaviour in a small neighbourhood around a single prediction may be approximately linear.

The procedure is four steps: perturb the input to generate nearby samples, ask the black-box model to predict each one, weight the samples by proximity to the original, then fit a weighted linear model. The coefficients of that little linear model **are** the explanation. It needs only a predict function, never model internals.


In [ ]:
!pip install lime
from lime.lime_tabular import LimeTabularExplainer

lime_explainer = LimeTabularExplainer(
    training_data=np.asarray(X_train_t),
    feature_names=feat_names,
    class_names=["did not survive", "survived"],
    mode="classification",
    random_state=RANDOM_STATE)

exp = lime_explainer.explain_instance(
    np.asarray(X_test_t)[i],
    model_only.predict_proba,
    num_features=8,
    num_samples=5000)

print("LIME explanation for the same passenger", i, ":\n")
for feature, weight in exp.as_list():
    direction = "toward survived" if weight > 0 else "toward died"
    print("  " + feature.ljust(34), str(round(weight, 4)).rjust(9),
          " ", direction)


LIME also has excellent domain-specific support that SHAP does not match: `LimeTextExplainer` highlights the words that drove a prediction, which is intuitive for content moderation and spam detection, and `LimeImageExplainer` highlights the superpixels that mattered, which is effective for debugging vision models. That multi-modality is its genuine strength.

### The stability problem

LIME's perturbations are random, so running it twice on the same prediction can produce different explanations. This matters enormously in practice: if a rejected loan applicant asks why, the answer should be the same on Tuesday as on Monday, and the same from two different support agents.


In [ ]:
import re
print("Running LIME three times on the SAME passenger, different seeds:\n")
for run in range(3):
    ex = LimeTabularExplainer(
        training_data=np.asarray(X_train_t), feature_names=feat_names,
        class_names=["died", "survived"], mode="classification",
        random_state=run).explain_instance(
            np.asarray(X_test_t)[i], model_only.predict_proba,
            num_features=4, num_samples=1000)
    top = [(re.sub(r"[^A-Za-z_]", "", f)[:14], round(w, 3))
           for f, w in ex.as_list()]
    print("  run", run + 1, ":", top)
print()
print("The rankings and weights shift between runs. Instability is worse")
print("with few perturbation samples, correlated features, or a locally")
print("complex decision boundary.")


**Mitigations:** raise `num_samples` (the default is 5000), fix a random seed for reproducibility, or average several runs. For high-stakes individual decisions where consistency is a legal or regulatory requirement, use SHAP instead.

### LIME versus SHAP

| | LIME | SHAP |
|---|---|---|
| Approach | Local surrogate linear model | Game theory, Shapley values |
| Speed | Fast, one linear fit per explanation | Slower, especially KernelSHAP |
| Determinism | Non-deterministic, random perturbations | Deterministic with exact methods |
| Theory | No uniqueness guarantee | Mathematically unique solution |
| Concept | Easy to explain: "we fit a line near this point" | More abstract |
| Non-tabular | Excellent text and image support | Weaker outside tabular and deep models |

In practice, SHAP has become the default for serious fairness auditing while LIME remains popular for quick exploration. Use both as a cross-check: **when LIME and SHAP agree on which features matter, you can trust the explanation more.**


---
# Section 8: Permutation importance

SHAP and LIME explain individual predictions. Permutation importance zooms out to a single global question: **which features does my model actually rely on?**

Shuffle one feature's values across the test set, which destroys any relationship between that feature and the target, then measure how much performance drops. A big drop means the feature mattered. Almost no drop means the model was not really using it.


In [ ]:
from sklearn.inspection import permutation_importance

result = permutation_importance(
    gbm, X_test, y_test, n_repeats=15, random_state=RANDOM_STATE,
    scoring="roc_auc")

perm = pd.DataFrame({
    "importance": result.importances_mean,
    "std": result.importances_std}, index=FEATURES
).sort_values("importance", ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(range(len(perm)), perm["importance"], xerr=perm["std"],
        color="steelblue")
ax.set_yticks(range(len(perm)))
ax.set_yticklabels(perm.index)
ax.invert_yaxis()
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Drop in AUC when this feature is shuffled")
ax.set_title("Permutation importance (test set, 15 repeats, error bars = std)")
plt.tight_layout(); plt.show()
print(perm.round(4))


**Use the test set, not training data**, because features that a model overfit to look important on training data and worthless on test data. And **repeat the shuffling** (`n_repeats`), because a single random shuffle is noisy and the standard deviation is your stability measure.

### Permutation importance versus built-in importance

Tree models offer a free built-in importance (Gini or gain) computed during training. It is convenient and it has a well-documented flaw: **it is biased toward high-cardinality features.** Let's prove that with a column that is pure noise.


In [ ]:
X_train_id = X_train.copy()
X_test_id = X_test.copy()
X_train_id["random_id"] = rng.random(len(X_train))     # pure noise
X_test_id["random_id"] = rng.random(len(X_test))

rf = Pipeline([
    ("preprocess", ColumnTransformer([
        ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                          ("scale", StandardScaler())]),
         NUMERIC + ["random_id"]),
        ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                          ("encode", OneHotEncoder(handle_unknown="ignore",
                                                   sparse_output=False))]),
         CATEGORICAL),
        ("pass", "passthrough", PASSTHROUGH)])),
    ("model", RandomForestClassifier(n_estimators=300,
                                     random_state=RANDOM_STATE))])
rf.fit(X_train_id, y_train)

names_id = [n.split("__")[-1] for n in
            rf.named_steps["preprocess"].get_feature_names_out()]
gini = pd.Series(rf.named_steps["model"].feature_importances_, index=names_id)

perm_id = permutation_importance(rf, X_test_id, y_test, n_repeats=10,
                                 random_state=RANDOM_STATE, scoring="roc_auc")
perm_s = pd.Series(perm_id.importances_mean,
                   index=list(X_train_id.columns))

print("A column of pure random noise was added. How do the two methods rank it?")
print()
print("  Built-in Gini importance of random_id:", round(gini["random_id"], 4),
      "| rank", int((gini > gini["random_id"]).sum()) + 1, "of", len(gini))
print("  Permutation importance of random_id:  ",
      round(perm_s["random_id"], 4),
      "| rank", int((perm_s > perm_s["random_id"]).sum()) + 1, "of", len(perm_s))
print()
print("Gini importance rewards random_id for splitting well on training data.")
print("Permutation importance on the test set correctly gives it roughly zero.")


| | Built-in (Gini or gain) | Permutation importance |
|---|---|---|
| When computed | During training, free | After training, extra cost |
| Model support | Tree models only | Any model |
| Cardinality bias | Yes, well documented | No |
| Data used | Training data, reflects overfitting | Test data, reflects generalization |
| Fairness auditing | Unreliable | The more trustworthy choice |

### The correlated features problem

Permutation importance has its own weakness. If two features carry the same information, shuffling one does not hurt because the other still provides the signal, so **both look less important than they are**. The classic example is height and weight: shuffle height alone and weight compensates.


In [ ]:
corr_check = X_test[["Fare", "Pclass", "SibSp", "Parch", "FamilySize"]].corr()
print("Feature correlations (FamilySize is SibSp + Parch + 1, so this is extreme):")
print(corr_check.round(2))
print()
print("Permutation importance of the correlated trio:")
for f in ["SibSp", "Parch", "FamilySize"]:
    print(" ", f.ljust(12), round(perm.loc[f, "importance"], 4))
print()
print("Each looks modest because the others cover for it. Fixes: permute")
print("correlated features together as a group, use conditional permutation,")
print("or use drop-column importance (retrain without the feature, expensive).")
print("SHAP handles correlation better through its interaction values.")


### Permutation importance for fairness screening

Run it **separately for each protected group**. If the importance rankings differ substantially between groups, the model may be treating them differently, and that is a flag for deeper SHAP investigation.


In [ ]:
per_group = {}
for g in ["female", "male"]:
    mask = (X_test["Sex"] == g).values
    r = permutation_importance(gbm, X_test[mask], y_test[mask], n_repeats=10,
                               random_state=RANDOM_STATE, scoring="roc_auc")
    per_group[g] = pd.Series(r.importances_mean, index=FEATURES)

pg = pd.DataFrame(per_group).sort_values("female", ascending=False).round(4)
pg["rank_female"] = pg["female"].rank(ascending=False).astype(int)
pg["rank_male"] = pg["male"].rank(ascending=False).astype(int)
print("Permutation importance computed separately per group:")
print(pg.head(8))
print()
print("Different rankings between groups mean the model leans on different")
print("evidence depending on who the passenger is. Worth investigating.")


---
# Section 9: Partial dependence and ICE plots

Permutation importance says which features matter. It says nothing about **how** they matter. Does raising a feature raise or lower the prediction? Is the relationship linear, stepped, or curved?

**Partial dependence plots**, introduced by Friedman in 2001, answer that. Pick a feature and a grid of values; for each grid value, force *every* sample to have that value, predict, and average. Plot the averages. The result shows how the model's average prediction moves as the feature varies.

How to read one:

- **Flat line**: no marginal effect, the model effectively ignores this feature
- **Rising line**: higher values increase the prediction
- **Step function**: a threshold effect, where predictions jump at a specific cutoff
- **U-shape or curve**: a nonlinear relationship


In [ ]:
from sklearn.inspection import PartialDependenceDisplay

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
PartialDependenceDisplay.from_estimator(
    gbm, X_test, features=["Age", "Fare", "Pclass"], ax=axes,
    grid_resolution=30)
for ax, name in zip(axes, ["Age", "Fare", "Pclass"]):
    ax.set_title("Partial dependence: " + name)
plt.tight_layout(); plt.show()


Sanity-check these against domain knowledge, which is one of the most useful things a PDP does. Fare rising with survival probability matches what we know: higher fares meant better-placed cabins. If a PDP contradicted the domain, that would be a signal that something is wrong with the model or the data.

### PDP hides heterogeneity, ICE reveals it

A PDP shows the **average** effect, and averages can conceal opposite effects that cancel out. If a feature raises predictions for one group and lowers them for another, the PDP may show a flat line and hide the interaction entirely.

**ICE plots** (Individual Conditional Expectation) draw one line per sample instead of one average. When all the lines run in the same direction, the PDP is trustworthy. When they diverge, there is an interaction the PDP is obscuring.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
PartialDependenceDisplay.from_estimator(
    gbm, X_test, features=["Age"], kind="both", ax=axes[0],
    grid_resolution=25, ice_lines_kw={"alpha": 0.15, "color": "steelblue"},
    pd_line_kw={"color": "firebrick", "linewidth": 3})
axes[0].set_title("Age: ICE lines (blue) with the PDP average (red)")

PartialDependenceDisplay.from_estimator(
    gbm, X_test, features=["Fare"], kind="both", ax=axes[1],
    grid_resolution=25, ice_lines_kw={"alpha": 0.15, "color": "steelblue"},
    pd_line_kw={"color": "firebrick", "linewidth": 3})
axes[1].set_title("Fare: ICE lines with the PDP average")
plt.tight_layout(); plt.show()


### ICE lines coloured by protected group

This is the fairness version of the plot, and it is the one worth putting in an audit report. Colour each ICE line by group: if the lines for different groups diverge, the feature affects those groups differently.


In [ ]:
grid = np.linspace(X_test["Age"].quantile(.05),
                   X_test["Age"].quantile(.95), 25)
sample_idx = X_test.sample(min(160, len(X_test)),
                           random_state=RANDOM_STATE).index

plt.figure(figsize=(9, 5))
for idx in sample_idx:
    row = X_test.loc[[idx]]
    replicated = pd.concat([row] * len(grid), ignore_index=True)
    replicated["Age"] = grid
    preds = gbm.predict_proba(replicated)[:, 1]
    colour = "firebrick" if row["Sex"].iloc[0] == "female" else "steelblue"
    plt.plot(grid, preds, color=colour, alpha=0.18, linewidth=1)

plt.plot([], [], color="firebrick", label="female")
plt.plot([], [], color="steelblue", label="male")
plt.xlabel("Age"); plt.ylabel("Predicted survival probability")
plt.title("ICE lines coloured by sex: two separated bands, not one population")
plt.legend(); plt.show()

print("The bands barely overlap. A single averaged PDP would have drawn one")
print("line through the middle of a gap that no actual passenger occupies.")


This plot shows, in one image, that the model is effectively running two different decision rules.

Two more PDP capabilities worth knowing: passing a **feature pair** such as `features=[("Age", "Fare")]` produces a 2D heatmap showing how two features jointly affect predictions, and the `PDPBox` library offers better categorical support. Note the computation cost is grid size times sample count per feature, so subsample to a few thousand rows on large datasets.

**The complete explainability toolkit:** SHAP explains why a specific prediction was made, permutation importance ranks which features matter, and PDP with ICE shows how each feature shapes predictions across its range. Use them together; each answers a question the others cannot.


---
# Section 10: Mitigation and documentation

You have measured the disparity and explained where it comes from. What can actually be done?

Interventions fall into three families by where they act:

- **Pre-processing**: fix the data. Reweight samples, resample underrepresented groups, or remove proxy features. Acts before training and works with any model.
- **In-processing**: fix the training. Add a fairness constraint or a penalty term to the objective. Most direct, but requires a model that supports it.
- **Post-processing**: fix the outputs. Adjust decision thresholds per group, as we did in Section 4. Easiest to retrofit onto a deployed model, and the most legally fraught, since it treats people differently by protected attribute on its face.

Let's measure a pre-processing intervention: reweighting so each sex-by-outcome cell carries equal total weight during training.


In [ ]:
from sklearn.metrics import accuracy_score

# Pre-processing: sample weights that equalize influence across sex-outcome cells
combo = X_train["Sex"].astype(str) + "_" + y_train.astype(str)
counts = combo.value_counts()
weights = combo.map(len(combo) / (len(counts) * counts)).values

reweighted = Pipeline([("preprocess", make_preprocessor()),
                       ("model", LogisticRegression(
                           max_iter=1000, random_state=RANDOM_STATE))])
reweighted.fit(X_train, y_train, model__sample_weight=weights)

rw_audit = audit.copy()
rw_audit["predicted"] = reweighted.predict(X_test)
rw_audit["probability"] = reweighted.predict_proba(X_test)[:, 1]

def gap(frame, metric="FNR", col="Sex"):
    m = group_metrics(frame, col)[metric]
    return round(float(m.max() - m.min()), 3)

rows = [
    {"model": "Baseline logistic regression",
     "accuracy": round(accuracy_score(y_test, audit["predicted"]), 4),
     "FNR gap (sex)": gap(audit), "FNR gap (class)": gap(audit, col="Pclass")},
    {"model": "Reweighted (pre-processing)",
     "accuracy": round(accuracy_score(y_test, rw_audit["predicted"]), 4),
     "FNR gap (sex)": gap(rw_audit),
     "FNR gap (class)": gap(rw_audit, col="Pclass")},
    {"model": "Sex removed (unawareness)",
     "accuracy": round(accuracy_score(y_test, blind_audit["predicted"]), 4),
     "FNR gap (sex)": gap(blind_audit),
     "FNR gap (class)": gap(blind_audit, col="Pclass")},
]
pd.DataFrame(rows).set_index("model")


On this dataset removing `Sex` produces the smallest error gap between sexes, because as Section 5 measured, the proxies here are weak. It also costs the most accuracy and does nothing for the disparity across passenger class. Reweighting sits between the two.

There is no row in that table that is simply "the fair one." Each buys a different thing at a different price, which is exactly why the choice has to be made explicitly and written down.

**Purpose-built tooling** exists and is worth using rather than hand-rolling: **Fairlearn** (Microsoft) provides fairness metrics plus mitigation algorithms and integrates with scikit-learn, **AI Fairness 360** (IBM) offers a large library of metrics and mitigations, and Google's **What-If Tool** gives interactive counterfactual exploration.

### The model card

Documentation is what turns an audit into accountability. A **model card** records what the model does, how it was evaluated, and crucially **which fairness definition you chose and why**, so that decision is visible and challengeable rather than buried in someone's notebook.


In [ ]:
def build_model_card(model, name, frame):
    overall = accuracy_score(frame["actual"], frame["predicted"])
    card = []
    card.append("MODEL CARD: " + name)
    card.append("=" * 64)
    card.append("Intended use : educational demonstration of fairness auditing")
    card.append("NOT for      : any real allocation, triage, or ranking decision")
    card.append("Training data: Titanic passenger manifest, 1912, n=" +
                str(len(X_train)))
    card.append("Known bias   : labels encode 1912 evacuation norms ('women and")
    card.append("               children first') and class-based berth location")
    card.append("")
    card.append("PERFORMANCE, DISAGGREGATED")
    card.append("  overall accuracy: " + str(round(overall, 3)))
    for col in ["Sex", "Pclass"]:
        m = group_metrics(frame, col)
        for g, row in m.iterrows():
            card.append("  " + (col + "=" + str(g)).ljust(16) +
                        "acc " + str(row["accuracy"]).ljust(7) +
                        "FNR " + str(row["FNR"]).ljust(7) +
                        "FPR " + str(row["FPR"]))
    card.append("")
    card.append("FAIRNESS DEFINITION CHOSEN: equalized odds (TPR/FPR parity)")
    card.append("  Rationale: in an allocation setting the harm falls on people")
    card.append("  the model fails to flag, so equal error rates across groups")
    card.append("  matters more than equal selection rates.")
    card.append("  Known tradeoff: predictive parity is NOT satisfied, and by")
    card.append("  the impossibility theorem it cannot be, since base rates")
    card.append("  differ sharply between groups.")
    card.append("")
    card.append("TOP FEATURES (permutation importance, test set)")
    for f, v in perm["importance"].head(5).items():
        card.append("  " + f.ljust(18) + str(round(v, 4)))
    card.append("")
    card.append("KNOWN PROXIES: Fare and Deck encode socioeconomic class;")
    card.append("  a model trained without Sex still predicts it at AUC " +
                str(round(proxy_auc, 2)))
    card.append("REVIEW: re-audit on any retrain; owner must be named.")
    return "\n".join(card)


print(build_model_card(logreg, "Titanic survival classifier v1", audit))


Alongside model cards, **datasheets for datasets** document where data came from, how it was collected, and what it does and does not represent. Both are cheap to produce and they are what an external auditor, a regulator, or your own successor will ask for first.

### What practitioners can actually do

- **Measure fairness explicitly.** Disaggregate every metric by protected group, and by intersections of groups. Nothing else in this list matters if you skip this.
- **Document the fairness definition you chose and why.** Surface it as a decision requiring stakeholder input, not a technical default.
- **Test for proxies.** Check whether removing a feature changes group outcomes, and use PDPs to reveal proxy effects.
- **Involve diverse perspectives.** A data scientist alone cannot evaluate whether a child welfare model is fair. You need domain experts, ethicists, legal input, and representatives of affected communities. Participatory design means including those communities in the design itself, not just the review.
- **Prefer simpler models in high-stakes decisions.** Interpretability is what lets people challenge and oversee a system. When COMPAS was a black box, nobody could contest it; once ProPublica could analyze it, accountability followed.
- **Watch who audits.** Internal teams have inherent conflicts of interest.

**Accept that mathematical perfection is impossible, and that progress is not.** You cannot satisfy every fairness definition at once. You can absolutely measure disparity, reduce it, document your choices, and build something better than what you started with.


---
# Section 11: Where you go from here

You have built, evaluated, secured, and now audited ML models. The last question is where to point all of it.

### Three career tracks

| Track | The role in one phrase | Core skills | Typical background |
|---|---|---|---|
| **Data Scientist** | The question answerer | Statistics, EDA, feature engineering, model selection, Python, SQL, heavy stakeholder communication | Statistics, mathematics, or domain expertise. A PhD is common but a strong portfolio substitutes |
| **ML Engineer** | The system builder | Software engineering plus ML: APIs, system design, distributed computing, testing | Software engineering or CS. Strong coding is non-negotiable |
| **MLOps Engineer** | The infrastructure specialist | Docker, Kubernetes, CI/CD, cloud platforms, monitoring, infrastructure as code | DevOps, SRE, or platform engineering |

A data scientist builds one model. An ML engineer makes it work at scale and reliably. An MLOps engineer builds the platform that lets a hundred data scientists deploy a hundred models. At a small company one person does all three; at a large one they are separate teams.

**Market reality:** data science attracts the most applicants and the most competition. ML engineering has high demand but demands real software engineering skill. MLOps has the highest demand-to-supply ratio and the least competition, because few people combine infrastructure depth with ML understanding.

### Portfolio strategy

ML is a "show me" field. Anyone can list scikit-learn on a resume, and hiring managers check GitHub repos before scheduling interviews. Kaggle competitions are not enough on their own, because they test model tuning on clean labeled data while real work is problem definition, collection, cleaning, and deployment.

**What a strong project contains:** a clear problem statement framed as a business or mission outcome rather than an algorithm name; real messy data rather than a pre-cleaned download; visible data work, since that is where 80% of the effort goes and hiring managers know it; rigorous evaluation with cross-validation, confusion matrices, and fairness checks rather than a single accuracy number; a README that explains the whole thing in five minutes; and a demo or deployment, even if that is just a Streamlit app.

**Common mistakes:** copying tutorials, no README, messy uncommented code, reporting only accuracy, skipping the data work, and no business context.

One honest note, since we have spent four modules on it: **Titanic is a teaching dataset and hiring managers recognize it instantly.** It was the right vehicle for learning because its problems are well understood and its bias is undeniable. It is the wrong vehicle for your portfolio.

**A three-project plan:** one classic end-to-end tabular project applying this course to a real problem; one domain-specific project in your area of expertise, which for a security professional means something like network anomaly detection or a phishing classifier; and one deployment-focused project that makes a model accessible through an API, container, or app. Write about each one, because blog posts multiply portfolio value and get read by hiring managers. One project a month gives you a strong portfolio in three.

### Certifications

Production ML lives in the cloud, so platform knowledge is a practical skill rather than a theoretical one. Certifications validate that you know SageMaker or Vertex AI, not just scikit-learn, and they help a resume pass automated screening at large organizations.

| Platform | Certification | Emphasis | Best for |
|---|---|---|---|
| **AWS** | Certified Machine Learning | Full lifecycle: SageMaker, S3, Glue, Athena, plus AI services like Rekognition and Comprehend | Largest market share, the safest default bet. Cloud Practitioner or Solutions Architect first |
| **GCP** | Professional ML Engineer | ML system design, Vertex AI, BigQuery ML, TFX pipelines, strong MLOps focus | ML engineering and MLOps roles |
| **Azure** | AI Engineer Associate | Broader: Azure ML, Cognitive Services, Bot Framework, enterprise integration | Microsoft-shop enterprises |

**Five rules:** certify on the platform your target employers actually use, so scan job postings first. Go deep on one rather than collecting three. Build while studying, implementing each service in a real project. Combine the certification with portfolio evidence, since "I am certified and here is the model I deployed" beats the credential alone. And plan to recertify, because these expire in one to three years and the platforms move fast. Budget four to eight weeks of focused preparation if your cloud fundamentals are solid.


Continuous learning is required in this field.


---
# Wrap-up

The audit, in order:

- **Overall accuracy looked fine**, and it was hiding everything. Disaggregating by sex, class, and their intersection revealed groups the model serves far worse than the headline number suggests.
- **False negative rates differed by a large multiple between groups**, which is the same shape of finding that made COMPAS a landmark case.
- **Every fairness definition we tried to satisfy broke another one**, exactly as the impossibility theorem predicts when base rates differ. That is not an engineering failure to route around; it is a values decision to make explicitly and document.
- **Deleting the protected attribute did almost nothing**, because a model trained without `Sex` still predicts it from the remaining features at high AUC. Proxies are the rule, not the exception.
- **SHAP, LIME, permutation importance, and PDP with ICE** each answered a different question, and the ICE plot coloured by sex made the disparity legible in a single image.
- **Mitigation traded a little accuracy for a meaningful reduction in disparity**, and the model card made the choice visible.

Three things to carry into your own work:

1. **Never report an aggregate metric alone.** Disaggregate by every group you can identify, and by their intersections.
2. **Fairness is a choice, not a calculation.** The mathematics tells you what is impossible; it cannot tell you what is right. Make the choice explicitly, document the reasoning, and involve the people affected.
3. **Explainability is the precondition for accountability.** A model nobody can interrogate is a model nobody can challenge, and in a high-stakes setting that is indefensible regardless of its accuracy.

You started by loading a CSV and finished by auditing a model for discrimination, explaining its decisions feature by feature, and knowing what you would say to a regulator who asked. Go build something, and audit it before you ship it.


---
# Try it yourself (optional)

**Exercise 1.** Repeat the false negative analysis using `Deck` (known vs Unknown) as the protected attribute. Since a recorded cabin is largely a wealth marker, does the model show a disparity between passengers whose cabin was recorded and those whose was not?

**Exercise 2.** Build the SHAP group-comparison table from Section 6 using `Pclass` instead of `Sex` as the grouping variable. Which feature shows the largest contribution ratio between first and third class?

**Exercise 3.** Find the decision threshold that minimizes the false negative rate **gap** between sexes, then report what it costs in overall accuracy. Would you ship it? Write two sentences justifying your answer.


In [ ]:
# Exercise workspace





---
## Exercise solutions


In [ ]:
# Exercise 1: cabin record as a socioeconomic proxy
audit["cabin_known"] = np.where(audit["Had_cabin_record"] == 1,
                                "recorded", "not recorded")
print(group_metrics(audit, "cabin_known"))
print()
print("FNR gap:", gap(audit, col="cabin_known"))
print("A recorded cabin is largely a wealth marker, so this is a")
print("socioeconomic disparity wearing a data-quality costume.")


In [ ]:
# Exercise 2: SHAP contributions by passenger class
first = (X_test["Pclass"] == 1).values
third = (X_test["Pclass"] == 3).values

by_class = pd.DataFrame({
    "mean |SHAP| 1st": np.abs(sv[first]).mean(axis=0),
    "mean |SHAP| 3rd": np.abs(sv[third]).mean(axis=0)}, index=feat_names)
by_class["ratio"] = (by_class["mean |SHAP| 1st"] /
                     by_class["mean |SHAP| 3rd"].replace(0, np.nan))
print(by_class.sort_values("ratio", ascending=False).head(6).round(4))


In [ ]:
# Exercise 3: the threshold that minimizes the FNR gap
results = []
for t in np.linspace(0.05, 0.95, 91):
    tmp = audit.copy()
    tmp["predicted"] = (tmp["probability"] >= t).astype(int)
    results.append({"threshold": round(t, 2),
                    "FNR gap": gap(tmp),
                    "accuracy": round(accuracy_score(tmp["actual"],
                                                     tmp["predicted"]), 4)})
res = pd.DataFrame(results)
best = res.loc[res["FNR gap"].idxmin()]
at_half = res.loc[res["threshold"] == 0.5].iloc[0]

print("Smallest FNR gap at threshold", best["threshold"],
      ": gap", best["FNR gap"], "accuracy", best["accuracy"])
print("For comparison at 0.5      : gap", at_half["FNR gap"],
      "accuracy", at_half["accuracy"])
print()
print("Note the degenerate solution: at a very low threshold the model")
print("predicts survival for nearly everyone, so both groups have an FNR")
print("near zero and the gap vanishes. A perfectly equal gap achieved by")
print("making the model useless is not fairness. Always read the accuracy")
print("column alongside the gap column.")
print()
print("Whether to ship it is not a question the code can answer. It depends")
print("on who is harmed by a missed case, how much accuracy the deployment")
print("can afford to lose, and whether a single global threshold is even the")
print("right lever. Write down your reasoning either way: that written")
print("rationale IS the deliverable.")
